## Bag of Words (BoW)
A machine learning model can't read text, it only understands numbers. Bag of Words converts each sentence into numbers by **counting how many times each word appears**.

Steps:
1. Make a list of all unique words in the data (the **vocabulary**)
2. For every sentence, count how many times each vocabulary word appears in it

It is called a "bag" because the **order of words is ignored**, we only care which words are present.

Example: vocabulary = [cold, hot, pizza, tasty]
- *"hot tasty pizza"* → [0, 1, 1, 1]
- *"cold pizza"* → [1, 0, 1, 0]

### Real life use case
**Spam filter in Gmail / SMS:** spam messages often have words like *free, win, prize, claim, call now*. BoW turns every message into word counts, so the model can learn that messages with many of these words are usually spam.

**Review analysis on Zomato / Amazon:** words like *tasty, fresh, fast* show up in good reviews and *cold, late, stale* show up in bad ones. Counting these words helps predict whether a review is positive or negative.

### Small example first

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

reviews = [
    "the pizza was hot and tasty",
    "the pizza was cold and the delivery was late",
    "tasty food and fast delivery",
]

cv = CountVectorizer()
X = cv.fit_transform(reviews).toarray()
# X
# rows = sentences, columns = vocabulary words (sorted alphabetically)
pd.DataFrame(X, columns=cv.get_feature_names_out())

,and,cold,delivery,fast,food,hot,late,pizza,tasty,the,was
0,1,0,0,0,0,1,0,1,1,1,1
1,1,1,1,0,0,0,1,1,0,2,2
2,1,0,1,1,1,0,0,0,1,0,0


In [4]:
# 'was' comes 2 times and 'the' comes 2 times in the 2nd sentence, so their count is 2
cv.vocabulary_  # word -> column number

{'the': 9,
 'pizza': 7,
 'was': 10,
 'hot': 5,
 'and': 0,
 'tasty': 8,
 'cold': 1,
 'delivery': 2,
 'late': 6,
 'food': 4,
 'fast': 3}

### Binary BoW
With `binary=True` we don't count, we only mark **1 if the word is present, 0 if not**.

This is useful when repeating a word doesn't add meaning, e.g. *"good good good phone"* is not 3 times better than *"good phone"*.

In [6]:
cv_binary = CountVectorizer(binary=True)
X = cv_binary.fit_transform(reviews).toarray()

# now 'was' and 'the' are just 1 instead of 2a
pd.DataFrame(X, columns=cv_binary.get_feature_names_out())

,and,cold,delivery,fast,food,hot,late,pizza,tasty,the,was
0,1,0,0,0,0,1,0,1,1,1,1
1,1,1,1,0,0,0,1,1,0,1,1
2,1,0,1,1,1,0,0,0,1,0,0


## Practical: SMS Spam Classification data
The dataset has **5572 SMS messages**, each labelled as:
- **ham** → normal message
- **spam** → unwanted/advertisement message

The file is tab separated (`	`) and has no header row, so we give the column names ourselves.

In [7]:
messages = pd.read_csv('SMSSpamCollection.txt', sep='	', names=["label", "message"])
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [8]:
# how many ham and spam messages
messages['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

### Data cleaning and preprocessing
For every message:
1. Keep only letters (remove numbers, punctuation, symbols)
2. Convert to lowercase
3. Split into words
4. Remove stopwords and stem the remaining words
5. Join the words back into one sentence

In [9]:
import re
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Code and
[nltk_data]     Debug\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

ps = PorterStemmer()

# set makes checking 'word in stop_words' much faster (we check it for every word of 5572 messages)
stop_words = set(stopwords.words('english'))

In [ ]:
corpus = []
for message in messages['message']:
    # [^a-zA-Z] means "anything that is NOT a letter" -> replace it with a space
    review = re.sub('[^a-zA-Z]', ' ', message)
    review = review.lower()
    words = review.split()

    clean_words = []
    for word in words:
        if word not in stop_words:
            stem_word = ps.stem(word)
            clean_words.append(stem_word)

    # join the words back into one sentence

    corpus.append(' '.join(clean_words))

In [14]:
# compare one message before and after cleaning
print("Before:", messages['message'][2])
print("After :", corpus[2])

Before: Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
After : free entri wkli comp win fa cup final tkt st may text fa receiv entri question std txt rate c appli


In [ ]:
# corpus[:10]

### Create Bag of Words
- `max_features=100` → keep only the **100 most frequent words** as columns (otherwise we get thousands of columns, one for every unique word)
- `binary=True` → 1 if the word is present, 0 if not

In [15]:
cv = CountVectorizer(max_features=100, binary=True)
X = cv.fit_transform(corpus).toarray()

# 5572 messages (rows) and 100 words (columns)
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(5572, 100))

In [16]:
# the 100 words that were selected
cv.get_feature_names_out()

array(['alreadi', 'ask', 'babe', 'back', 'call', 'care', 'cash', 'claim',
       'co', 'come', 'contact', 'da', 'day', 'dear', 'dont', 'even',
       'feel', 'find', 'free', 'friend', 'get', 'give', 'go', 'good',
       'got', 'great', 'gt', 'happi', 'hey', 'hi', 'home', 'hope', 'im',
       'know', 'last', 'later', 'leav', 'let', 'like', 'lor', 'love',
       'lt', 'make', 'meet', 'messag', 'min', 'miss', 'mobil', 'morn',
       'msg', 'much', 'need', 'new', 'night', 'number', 'oh', 'ok', 'one',
       'phone', 'pick', 'pl', 'pleas', 'prize', 'realli', 'repli',
       'right', 'said', 'say', 'see', 'send', 'servic', 'sleep', 'sorri',
       'still', 'stop', 'take', 'tell', 'text', 'thank', 'thing', 'think',
       'time', 'today', 'tomorrow', 'tri', 'txt', 'ur', 'use', 'wait',
       'want', 'wat', 'way', 'week', 'well', 'win', 'work', 'www', 'ye',
       'yeah', 'year'], dtype=object)

### `cv.vocabulary_`
It is a dictionary of **word → column number**. It tells us which column of `X` belongs to which word.

Careful: the number is **not the count** of the word, it is only the column position. Columns are in alphabetical order, so `'alreadi'` is column 0, `'ask'` is column 1, and so on.

In [ ]:
# # np.int64(18) is just the number 18 (numpy's way of showing it)
# cv.vocabulary_

In [ ]:
# # which column is 'free'?
# col = cv.vocabulary_['free']
# col

In [ ]:
# # message 2 is the "Free entry in 2 a wkly comp..." spam message
# # it has the word 'free', so row 2 has 1 in the 'free' column
# X[2, col]

In [17]:
# view it as a table (pandas shows only some of the 100 columns)
pd.DataFrame(X, columns=cv.get_feature_names_out()).head()

,alreadi,ask,babe,back,call,care,cash,claim,co,come,...,wat,way,week,well,win,work,www,ye,yeah,year
0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# # most values are 0, because one SMS has only a few of the 100 words
# X

### N-Grams
So far every column was a single word (**unigram**). With N-grams, a group of words together also becomes a column:
- **bigram** (2 words): *"free entri"*, *"call later"*
- **trigram** (3 words): *"sorri call later"*

Why? Single words lose meaning. *"not good"* has the word *good* in it, but the bigram **not good** keeps the real meaning.

`ngram_range=(2, 3)` means use bigrams and trigrams. `(1, 2)` would mean single words + bigrams.

In [20]:
cv = CountVectorizer(max_features=100, binary=True, ngram_range=(1,2))
X = cv.fit_transform(corpus).toarray()

X.shape

(5572, 100)

In [21]:
# now the columns are 2 and 3 word phrases
cv.get_feature_names_out()

array(['alreadi', 'ask', 'babe', 'back', 'call', 'care', 'cash', 'claim',
       'co', 'come', 'da', 'day', 'dear', 'dont', 'even', 'feel', 'find',
       'free', 'friend', 'get', 'give', 'go', 'good', 'got', 'great',
       'gt', 'happi', 'hey', 'hi', 'home', 'hope', 'im', 'know', 'last',
       'later', 'leav', 'let', 'like', 'lor', 'love', 'lt', 'lt gt',
       'make', 'meet', 'messag', 'min', 'miss', 'mobil', 'morn', 'msg',
       'much', 'need', 'new', 'night', 'number', 'oh', 'ok', 'one',
       'phone', 'pick', 'pl', 'pleas', 'prize', 'realli', 'repli',
       'right', 'said', 'say', 'see', 'send', 'servic', 'sleep', 'sorri',
       'still', 'stop', 'take', 'tell', 'text', 'thank', 'thing', 'think',
       'time', 'today', 'tomorrow', 'tri', 'txt', 'ur', 'use', 'wait',
       'want', 'wat', 'way', 'week', 'well', 'win', 'work', 'www', 'ye',
       'yeah', 'year'], dtype=object)

In [ ]:
# X